In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
BRONZE_PATH = "abfss://bronze@usgridenergypipeline.dfs.core.windows.net/"
SILVER_PATH  = f"abfss://silver@usgridenergypipeline.dfs.core.windows.net/silver_fuel_type_data"
CHECKPOINT = "abfss://bronze@usgridenergypipeline.dfs.core.windows.net/_checkpoints/fuel_type_data"

DIM_PATH = f"abfss://silver@usgridenergypipeline.dfs.core.windows.net/dim_fuel_type"

SILVER_TABLE = "us_grid_energy_pipeline_databricks.usgrid.silver_fuel_type_data"

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS usgrid")

DataFrame[]

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
        region_code    STRING,
        fuel_type_code STRING,
        period         TIMESTAMP,
        value_in_megawatts FLOAT
    )
    USING DELTA
    LOCATION '{SILVER_PATH}'
    PARTITIONED BY (region_code, fuel_type_code)
""")

DataFrame[]

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS usgrid.dim_fuel_type (
        fuel_type_code STRING,
        fuel_type_name STRING
    )
    USING DELTA
    LOCATION '{DIM_PATH}'
""")

DataFrame[]

In [0]:
df_raw = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT + "/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("multiLine", "true")
    .load(BRONZE_PATH + "*/fuel_type_data/*/*/*.json")
)

In [0]:
df_exploded = (
    df_raw
    .withColumn("row", F.explode("raw_data.response.data"))
    .select(
        F.col("region").alias("region_code"),
        F.col("row.fueltype").alias("fuel_type_code"),
        F.col("row.period").alias("period"),
        F.col("row.value").alias("value_in_megawatts"),
        F.col("row.type-name").alias("fuel_type_name")
    )
)

In [0]:
df_typed = (
    df_exploded
    .withColumn("period",F.to_timestamp("period", "yyyy-MM-dd'T'HH"))
    .withColumn("value_in_megawatts", F.col("value_in_megawatts").cast("float"))
    .dropDuplicates(["region_code", "fuel_type_code", "period"])
)

In [0]:
def merge_to_silver(micro_batch_df, batch_id):

    # 1. Dim table
    dim_df = (
        micro_batch_df
        .select("fuel_type_code", "fuel_type_name")
        .dropDuplicates(["fuel_type_code"])
    )
    dim_df.createOrReplaceTempView("dim_updates")

    spark.sql("""
        MERGE INTO us_grid_energy_pipeline_databricks.usgrid.dim_fuel_type AS target
        USING dim_updates AS source
        ON target.fuel_type_code = source.fuel_type_code
        WHEN NOT MATCHED THEN INSERT *
    """)

    # 2. Silver table
    silver_df = micro_batch_df.drop("fuel_type_name")
    silver_df.createOrReplaceTempView("silver_updates")

    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING silver_updates AS source
        ON target.region_code = source.region_code
        AND target.fuel_type_code = source.fuel_type_code
        AND target.period = source.period
        WHEN MATCHED AND target.value_in_megawatts != source.value_in_megawatts
            THEN UPDATE SET target.value_in_megawatts = source.value_in_megawatts
        WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
(
    df_typed
    .writeStream
    .format("delta")
    .foreachBatch(merge_to_silver)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination()
)

In [0]:
# dbutils.fs.rm(CHECKPOINT + "/schema", recurse=True)

In [0]:
# # 1. Drop table
# spark.sql("DROP TABLE IF EXISTS us_grid_energy_pipeline_databricks.usgrid.silver_fuel_type_data")

# # 2. Delete ADLS data
# dbutils.fs.rm("abfss://silver@usgridenergypipeline.dfs.core.windows.net/silver_fuel_type_data", recurse=True)

# # 3. Delete entire checkpoint folder (not just schema)
# dbutils.fs.rm("abfss://bronze@usgridenergypipeline.dfs.core.windows.net/_checkpoints/fuel_type_data", recurse=True)

In [0]:
# spark.sql("SHOW TABLES IN us_grid_energy_pipeline_databricks.usgrid").show()

In [0]:
# spark.sql("SELECT * FROM us_grid_energy_pipeline_databricks.usgrid.silver_fuel_type_data LIMIT 10").display()

In [0]:
# # spark.sql("""
#     SELECT fuel_type_code, region_code, COUNT(*) as row_count
#     FROM us_grid_energy_pipeline_databricks.usgrid.silver_fuel_type_data
#     GROUP BY fuel_type_code, region_code
#     ORDER BY row_count DESC
# """).display()

In [0]:
# # spark.sql("""
#     SELECT region_code, COUNT(DISTINCT fuel_type_code) as fuel_types, COUNT(*) as total_rows
#     FROM us_grid_energy_pipeline_databricks.usgrid.silver_fuel_type_data
#     GROUP BY region_code
# """).display()

In [0]:
# %sql
# select region_code from usgrid.silver_fuel_type_data sftd
# join usgrid.dim_fuel_type
# on usgrid.silver_fuel_type_data.fuel_type_code = usgrid.dim_fuel_type.fuel_type_code

In [0]:
# %sql
# select distinct dft.fuel_type_code, dft.fuel_type_name from usgrid.silver_fuel_type_data ft
# join usgrid.dim_fuel_type dft